# RescueEye — Casualty Detection Training
YOLO11s trained on 10,542 fallen/casualty images. GPU accelerated.

**Before running:** Make sure GPU is enabled — Notebook settings → Accelerator → GPU T4 x2

In [1]:
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — enable it first!')
print('Torch:', torch.__version__)

CUDA available: True
Device: Tesla T4
Torch: 2.10.0+cu128


In [2]:
# Install YOLO11
!pip install ultralytics -q
from ultralytics import YOLO
print('Ultralytics ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.8 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[

In [3]:
import os, zipfile
from pathlib import Path

# Find the uploaded dataset
dataset_zip = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.zip'):
            dataset_zip = os.path.join(root, f)
            print('Found zip:', dataset_zip)
            break
    if dataset_zip:
        break

if dataset_zip:
    print('Extracting...')
    with zipfile.ZipFile(dataset_zip, 'r') as z:
        z.extractall('/kaggle/working/dataset')
    print('Extracted to /kaggle/working/dataset')
else:
    # Already extracted
    print('No zip found, scanning /kaggle/input...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data.yaml' in files:
            dataset_zip = root
            print('Dataset folder:', root)
            break

No zip found, scanning /kaggle/input...
Dataset folder: /kaggle/input/datasets/prinsss/rescueeye-casualty


In [4]:
import yaml

# Find data.yaml and fix the path to point to Kaggle working dir
yaml_path = None
for root, dirs, files in os.walk('/kaggle/working/dataset'):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'data.yaml' in files:
            yaml_path = os.path.join(root, 'data.yaml')
            break

print('data.yaml:', yaml_path)

# Fix the path inside data.yaml
dataset_root = str(Path(yaml_path).parent)
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['path'] = dataset_root
cfg['train'] = 'images/train'
cfg['val']   = 'images/val'
cfg['test']  = 'images/test'

fixed_yaml = '/kaggle/working/data.yaml'
with open(fixed_yaml, 'w') as f:
    yaml.dump(cfg, f)

print('Fixed data.yaml:')
print(open(fixed_yaml).read())

data.yaml: /kaggle/input/datasets/prinsss/rescueeye-casualty/data.yaml
Fixed data.yaml:
names:
- casualty
nc: 1
path: /kaggle/input/datasets/prinsss/rescueeye-casualty
test: images/test
train: images/train
val: images/val



In [5]:
# Count images
for split in ['train', 'val', 'test']:
    path = Path(dataset_root) / 'images' / split
    count = len(list(path.glob('*.jpg')) + list(path.glob('*.png'))) if path.exists() else 0
    print(f'  {split}: {count} images')

  train: 2980 images
  val: 360 images
  test: 375 images


In [ ]:
# TRAIN
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data        = fixed_yaml,
    imgsz       = 1280,
    batch       = 16,
    epochs      = 150,
    patience    = 30,
    warmup_epochs = 5,
    optimizer   = 'AdamW',
    lr0         = 0.001,
    lrf         = 0.01,
    weight_decay= 0.0005,
    # Augmentation
    degrees     = 45.0,
    translate   = 0.1,
    scale       = 0.5,
    flipud      = 0.5,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.15,
    copy_paste  = 0.3,
    hsv_h       = 0.02,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    # Output
    name        = 'casualty_v1',
    project     = '/kaggle/working/runs',
    save        = True,
    save_period = 10,
    plots       = True,
    device      = 0,
    workers     = 2,
    seed        = 42,
)

print('\nTraining complete!')
print(f'mAP@0.5:      {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')
print(f'mAP@0.5:0.95: {results.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')

Ultralytics 8.4.68 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=casualty_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=

In [ ]:
# Export best.pt to ONNX
best_pt = '/kaggle/working/runs/casualty_v1/weights/best.pt'
model   = YOLO(best_pt)
onnx    = model.export(format='onnx', imgsz=1280, opset=13, simplify=True, dynamic=False)
print('ONNX exported to:', onnx)

In [ ]:
# Show training results
from IPython.display import Image, display
import glob

for img in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    matches = glob.glob(f'/kaggle/working/runs/casualty_v1/{img}')
    if matches:
        print(img)
        display(Image(matches[0]))

In [ ]:
# Package outputs for download
import shutil
shutil.copy('/kaggle/working/runs/casualty_v1/weights/best.pt',   '/kaggle/working/casualty_best.pt')
shutil.copy('/kaggle/working/runs/casualty_v1/weights/best.onnx', '/kaggle/working/casualty_best.onnx')
print('Download these from the Output tab on the right:')
print('  casualty_best.pt   (PyTorch weights)')
print('  casualty_best.onnx (ONNX for RescueEye)')